# Greedy generation with a KV cache: prefill once, then decode

Without a cache, generating token $t$ recomputes keys and values for all $t - 1$ earlier tokens. With one, the prompt is processed once (**prefill**), and every later step pushes only the newest token through the model (**decode**), appending its key and value to the cache.

Write `generate(prompt, n_new, E, Pos, Wq, Wk, Wv, Wo, W1, W2)` in PyTorch for this tiny one-layer, one-head model.

## Model (float64; projections are $hW$, no biases)
For a token $t$ at position $i$:
1. $x = E[t] + \text{Pos}[i]$, with `E`: `(V, D)` token embeddings and `Pos`: `(max_len, D)` position embeddings.
2. $q = xW_q$, $k = xW_k$, $v = xW_v$ (all `(D, D)`). Causal attention over positions $\le i$: $a = \mathrm{softmax}(q K^\top / \sqrt{D})\,V$, then $h = x + a W_o$.
3. $h \leftarrow h + \mathrm{relu}(h W_1) W_2$, with `W1`: `(D, F)`, `W2`: `(F, D)`.
4. $\text{logits} = h E^\top$ (tied output embedding), shape `(V,)`.

## Generation
- `prompt`: int64 `(T_0,)` with $T_0 \ge 1$. `n_new` $\ge 1$. $T_0 + n_{\text{new}} \le$ `max_len`.
- **Prefill:** run the whole prompt in one batched pass (causal mask), store its keys and values, and take the logits at the last prompt position.
- **Decode:** the next token is the argmax of those logits (smallest index on ties). Feed **only** that token through steps 1-4, using and extending the cache, to get the next logits. Repeat until `n_new` tokens have been produced. The last produced token is never fed back.

## Output
A tuple `(tokens, logits, K)`:
- `tokens`: int64 `(n_new,)`, the generated tokens.
- `logits`: `(n_new, V)`, row $s$ holds the logits that produced `tokens[s]`.
- `K`: `(T_0 + n_{\text{new}} - 1, D)`, the final key cache.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def generate(prompt, n_new, E, Pos, Wq, Wk, Wv, Wo, W1, W2):
    """Return (tokens (n_new,) int64, logits (n_new, V), key cache (T0 + n_new - 1, D))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [0], "dtype": "int64"}, 2, {"__nd__": [[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0.0], [0.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0, 0], "dtype": "int64"}, {"__nd__": [[1.0, 0.0, -1.0], [1.0, 0.0, -1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [1.0, 0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [1, 2], "dtype": "int64"}, 2, {"__nd__": [[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0.0], [0.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [2, 2], "dtype": "int64"}, {"__nd__": [[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[0.0, 1.0], [-1.0, 0.0], [-1.0, 0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [3, 1, 4], "dtype": "int64"}, 5, {"__nd__": [[0.5532605888887387, 0.21760060921947388, -0.057989987866526184, -2.3189360947215656, 0.4314941677088505, -2.126279784450882], [0.9099212170226342, 0.6059655730064136, 0.8300566485784159, 0.8276983437153878, 0.2985144698332214, -0.5350014137339273], [-0.3070622870057959, 1.5080722715721342, -0.5822293778059019, -0.22812380385128453, -0.7245145044162855, -0.5172493374907822], [-0.30655788195456973, 0.2565516458675206, -0.2938560550257995, -0.3547084649887684, -0.6168151525371501, 0.08958446690608873], [-1.3441660008066356, 0.052184899307776686, 1.3100257323485311, -0.7669128788338543, -0.04539173803348671, 2.817209016484243], [-0.9926744097609564, -1.621141524211868, -0.42680913157992273, 1.8069379803781687, 2.035115511784091, -1.2159872631305197], [0.5701405884703373, 0.004521431343134918, -0.5883707554353481, -1.9456978173547614, 1.0245839797071592, -0.8605383067300788], [0.29900768417420936, 0.08189396317057414, 1.8701812688972848, 1.1759947894135871, -1.1553360632608776, -0.2623951558776075]], "dtype": "float64"}, {"__nd__": [[0.3110542658186933, -0.39990448928440164, -0.11768649956008068, 0.06827274021061877, -0.48474784760471146, -0.370637443931036], [-0.5112231012351282, 0.08609276363583922, -0.16845434069393836, 0.27859448629364864, 0.034138276069569755, 0.1716051760129469], [-0.025102753174482025, 0.22557713857369896, -0.03099514716495763, -0.29466046524598477, 0.35483051467472093, 0.2940024442964846], [0.6800079641643436, -0.042660279447610056, -0.17397008489402283, -0.4341127508187548, -1.157783254414943, -0.3925195020261963], [0.14604556513530131, -0.26565934241092876, -0.7784471997248801, 0.18247791235464164, -0.17749725243968703, -0.10830122805835589], [-0.1294318504964284, -0.04040951868110347, 0.14233322308678498, -0.5203549953265403, -0.06813523281590272, -0.15257673321520535], [0.3410174161000202, -0.3105892059921359, 0.4912234336292887, 0.18240995925669357, 0.48739301591346396, -0.21134561253051698], [0.24178563456686442, 0.10893120005147075, -0.44930110607373325, -0.07827536916846163, -0.13580889964303594, -0.289846129889281], [-0.23044486686093, -0.4319480429835773, -0.14015216375344489, 0.1002408908938511, -0.23365158992629323, -0.46894559591564344], [0.3135990255973702, -0.11124438368049472, 0.008227418288790072, -0.0701830240247237, -0.16203555787028018, 0.18052762193667335]], "dtype": "float64"}, {"__nd__": [[-0.26495310740834405, 0.014363647150386107, -0.7224709226894472, -0.8193828960829407, -0.19477847768504572, 0.9553357517865533], [0.0906312971726383, 0.9668150300644179, 0.2260922749748064, 0.2267296539313201, -0.2916618484010761, -0.011208783259086851], [0.8956428599135954, 0.5925947902411222, -0.3130946451761318, -0.23490847998992007, 0.6686208714749875, -0.2573926317313178], [0.4598702486688066, -0.45478111871920673, 0.3140082970313042, 0.7631010423334039, 0.417503065392336, 0.12172199692002993], [0.07019999148677628, -0.042278102755672134, -0.011530669779205248, 0.5894112177830613, 0.57695245713104, 0.20390608479802183], [-0.6296068038822974, -0.2603293207663655, 0.14870044241341093, 0.02574775954332128, -0.6764832896797969, 0.3320046660548398]], "dtype": "float64"}, {"__nd__": [[1.2994154915752452, -0.6132704598735359, 0.031886876049058197, 0.80989428310521, -0.20867405180337587, -0.20387087453459798], [0.08478761549371581, 0.28349080693353335, -0.20559785550988655, 0.4899071771411677, -0.9524985108973736, 0.3277109286224365], [0.18438563293424487, -0.07945449406142631, 0.19368371576269688, 0.02393889659188959, 0.664909335043508, -0.23085582908563795], [-0.02070946521966988, -0.6938274406936995, -0.5623880878168004, -0.12049146343783547, -0.7780141748963129, -0.42364404596969907], [-0.6630105883368398, -0.2468009740973975, 0.05871919714802957, 0.47176849464314086, -0.9703039408144716, -0.10016948523028213], [0.4328536527742852, 0.6900651420864516, 0.7030354751309926, -0.7171632922482637, -0.25656986304423174, 0.2820937446921854]], "dtype": "float64"}, {"__nd__": [[0.4693617449384219, 0.1410094141448737, 0.13279320164481917, 0.12054928571569677, -0.383680393168534, 0.08064521285511239], [0.406118410077343, -0.16953519188423288, 0.5864176606100334, -1.4210672778425912, -0.16325584686658473, 0.850110117753074], [0.4367768286715696, 0.7220864264369719, 0.07306259900307331, -0.5908620730929616, -0.25522550769547814, 0.761098593317215], [-0.9182907720109619, -0.4768158622562924, -0.8569672059145028, -0.2668174822592829, -0.2106189950287257, 0.28996817024970967], [-0.411957710714334, 0.8567500151622259, 0.3238636949202844, -0.23545258493775217, -0.4955322669150621, -0.7489283147867263], [0.06168068707445052, 0.9918573191064075, -0.08023232256487499, -0.5130716680883082, -0.027940823269131814, -0.25695716966595356]], "dtype": "float64"}, {"__nd__": [[-0.4507348569314965, -0.9467605365304715, -0.026708048634337, 0.7444473792602774, 0.6157412658566772, 0.49378647300142875], [0.10744882708947062, 0.23873309218953742, 0.5085921479298426, -0.023879965051455263, 0.1446598092318838, -0.19571195267893776], [0.5703916042921024, -0.45910417935956066, 0.34505001703072385, 0.4325099145209079, 0.4230276527501849, 0.36069575466929693], [-0.22705812849531526, -0.27685251887128864, -0.20070660414000613, -0.1820637842171048, -0.5575307347246861, -0.21722154444802794], [-0.3126900840953839, -0.4684359924365673, 0.18783745678843225, -0.3721272586611385, 0.13124968666998146, 0.74338815626236], [0.8514350293496906, 0.01702126132139234, 0.5190857497872426, 0.13332854101439645, 1.2054798045000619, 0.8327986428207202]], "dtype": "float64"}, {"__nd__": [[0.5683981706176172, -0.06917307065226996, -0.11619586876489398, 0.2831720122870191, -0.0069700670523425426, -0.1143420296002284, 0.35783861871562367, -0.6167802692213107, -0.9946492931818304, -0.5016416718246672, -0.5723950094868783, -0.03400493677329704], [-0.4296342820707714, -0.09136547971584204, -1.00879637095786, 0.7449641611009274, 0.20534872892306685, -0.4277036249510701, 0.16476920430132547, 0.6242956155924095, 0.42875606333973876, -0.03918194115305441, 0.31480505291712196, 0.450769533796196], [-0.6013912018406024, -0.7177159221223561, -0.2929432926549633, 0.5004769802871523, 0.7481613505871244, 0.07315681989808337, 0.7627829393165564, 0.5080646408882336, -0.029850493109378783, -0.36866641745600937, -0.3173734478587349, 0.046915017618395796], [0.7128360244232012, 0.4285755176317826, -0.6501618387736307, -0.6115686318079506, 0.4154525960606271, 0.10513514313459588, -0.46983099467327516, 0.3475307084287639, 0.6942040269255958, 0.40857376514544197, 0.06107902363478306, 0.6802979428913483], [-0.18677903455183426, -0.2902249397998787, -0.45648946493170683, 0.9608516318622081, 1.0766499831802723, -0.012140731867006491, 0.30703172412371493, 0.23700932931366703, 0.4803792348070934, 0.6098641950177002, 0.3527409895936315, -0.03602580736653829], [-0.2501618529130536, -0.5024631842508256, -1.3047291067451863, -0.037882754434889955, -0.408588312530154, 0.02571053804847279, -0.04238161906582075, -0.506244214069944, -0.24169178052683293, 0.21827776364442472, 0.09485888298821514, -0.43322591904069013]], "dtype": "float64"}, {"__nd__": [[-0.36893371271440406, 0.29642697021648734, 0.12928730583976925, 0.16462867775147622, -0.072374280005162, 0.38145537937110147], [0.05706017860985859, 1.2785561048809706, -0.2272633054918695, -0.34356849400772155, 0.8383206229873289, -0.027042717302653254], [-0.3820484533058104, 0.22638842981945698, -1.1531260619895687, 0.23247060025087715, 0.01799563825805633, 0.14247106713984264], [-0.3951535111923351, 0.6671791469351137, 0.16539410330003312, -0.0020449989289683676, 0.18911386217953238, -0.41135895691998986], [-0.057273177001874565, -0.7457219317096269, -0.5466829078524467, 0.15333048749973116, 0.06477163891736408, 0.3771887081799816], [-0.3269598556867811, -0.05105603788529454, -0.33973143482459556, 0.13197900665882137, 0.3950381982015876, 0.5379965240065855], [0.2882252335831251, -0.3949294697180448, 0.10658160459692957, -0.5182442505333023, -0.9607244319597289, 0.08657890156266898], [0.04104321664756129, 0.3705501067890136, 0.4106636330230797, -0.6194052553957058, -0.4573330921058632, -0.006521294841003315], [0.03182857883751955, 0.19958419321372417, -0.6589055941238099, -0.13170948301705743, 0.31445035236783614, 0.9414159152545104], [-0.6344133530880939, 0.802632210972514, -0.1509690848265576, -0.7301618380963739, 0.6567395008644509, -0.5430254181959883], [-0.5936264057210116, 0.34915667339875484, 0.5074717144128703, 0.022713407558615756, -0.3720051224363784, -0.7574345144206723], [0.04133571806989796, -1.1818521113109721, -0.741155417648379, 0.5154199965034012, 0.2948837174841876, 0.5947373532453915]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [4, 4, 4, 4, 4], "dtype": "int64"}, {"__nd__": [[-0.5778798538924117, -3.380248703746521, 0.28017152835519044, 1.2301176477039846, 17.442554930920455, -7.699925612756932, 1.3500591826165926, -2.110451158703545], [-3.7383729189430994, -1.6891818190132364, -3.1968194826164744, 0.5262600486557978, 14.707628849133359, -8.327694813332766, -2.4299112580516993, 3.7107849872935987], [-1.3081418577002983, -1.7231280673356644, 0.40304830283623105, 0.5699730657567736, 11.347277282170584, -4.1646977096998805, 0.46612091740748274, -1.6055406472786993], [-1.6554944731577805, -4.1876866865241, -1.3453100999677217, 1.2912652469714567, 18.549050808996064, -7.755973245951113, 0.26409078639914974, -0.8726528462649159], [-3.5762526659580023, -4.375191310152781, -3.3243833813747665, 0.4660063878441749, 18.12930721803424, -4.294045365093178, 0.11501376421053022, -2.3122402186978652]], "dtype": "float64"}, {"__nd__": [[0.5324306464406345, 0.26596111758897206, -0.15127774701958197, -0.36004864629744227, 1.2257783029269138, 0.19951852377786972], [0.2979820299919506, -1.201327298207311, -0.8595441328667394, 0.962036385345445, -1.392726186437451, -0.6117304330461886], [-0.3563407099506503, 3.623967838341787, 2.9494366410941164, -2.8996194015508454, 0.5990286539060347, 1.3712942026627915], [1.2193887390409595, 3.1231869333214246, 2.5063350952392796, -2.6678407327639446, 2.3646525117279955, 1.1895772549422565], [-0.14449907212397098, 2.9618465999972643, 2.328693350102151, -3.0396699566684475, 0.7827469174508332, 1.0856709614661362], [-0.39069931452810736, 3.5515859948634714, 2.822497783066193, -2.9623518800805715, 1.6899766093587298, 1.2773884804914495], [-0.14628112015238787, 2.4935021262284858, 2.5566985839754635, -2.485798199056655, 1.010418140518811, 0.6424459013791275]], "dtype": "float64"}]}}, {"a": [{"__nd__": [0], "dtype": "int64"}, 1, {"__nd__": [[0.37670143252552796, -1.1753494116182133, 0.43693093305720426, -1.8453704739528716], [0.6581871650214526, 1.3261683934876929, -1.0405556845107033, 0.986209741497253], [-0.0029098795250363098, -0.20389078034935193, -2.0172003657999125, -0.20042432191697002], [0.8230373717303825, -1.7599184629959224, -0.5283723573590631, -0.7507297210471999], [-1.6745860195820719, 0.997359557033254, -0.02731258298736381, -0.004999852401579111]], "dtype": "float64"}, {"__nd__": [[0.2809331563151366, -0.23949545837443006, 0.9625776408098194, -0.15160407434287537], [-0.3623539249977419, 0.17740582294960194, 0.10200109771566071, -0.2988735778013455], [-0.28102017594607803, 0.24075979442496404, 0.08006122936514135, 0.11723514082648412], [0.2548633795523416, -0.1865297391887307, 0.3740521900964544, 0.0051287364536565325]], "dtype": "float64"}, {"__nd__": [[0.35543419869393517, -0.4622382757292728, 0.3734147388963546, 0.13855879984543262], [1.2381036661870513, -0.07500419200713787, 0.7661029190307985, 0.3215338965601267], [0.7884265832210201, 0.12433667719671553, 0.24733540422751688, -1.100932440279034], [-0.6307355084641036, 0.5299655434243931, 0.21985155911106152, 0.3944618520692279]], "dtype": "float64"}, {"__nd__": [[-0.12201832208260292, 0.4009204970787009, -0.16915465208347624, -0.019479813180565773], [0.6162496623134178, 0.5475357232394982, -0.20865591456506744, 0.034572293636906566], [0.12725439990455814, -0.06144533074696217, -0.041055370982313814, -0.4615595096885468], [0.2921419153376601, -0.512962704724402, -0.47232620995215024, 0.6427991270821478]], "dtype": "float64"}, {"__nd__": [[0.9205687337540135, 0.9302497665986922, 0.18484764743306795, -0.2589764625451857], [0.37883806328457714, 0.7821778298548078, 0.06686867980751979, -0.19289341967943602], [-0.8234361673840708, -0.8336612429902938, 0.8650504107347405, -0.7248230445081958], [-0.32630375458199445, -0.14267343445023725, 0.12826144387862531, -0.6406539016014626]], "dtype": "float64"}, {"__nd__": [[0.28905814152980813, 0.29253415759715184, 0.03971300890513885, -0.3064293973992806], [-0.1817801548231359, -0.9592986306686175, 0.3141948211645769, -0.2600935063876689], [-0.6824628548371632, -0.01920792659079282, 0.24603806167917028, 0.17612899755266723], [0.30824538263394596, 0.5789303230681876, -0.26621247473781506, 0.08913358108479709]], "dtype": "float64"}, {"__nd__": [[0.12136052822667058, -0.732827019128536, 0.5354724252706559, -0.17229510807072335, 0.11351620069614417, 0.13870984590172147, -0.4175951070178612, 0.3973178365229531], [0.47343416535598215, 0.07283736683961813, -1.4557473096297961, 0.6664647417616656, -0.08654834544084326, -0.33543765899861483, -0.3046570036615591, -0.5533933428596335], [-0.1972575661113017, 0.29420227923466263, -0.5653100246566451, 0.09111454728654196, -0.3106858906346095, -0.1427813727326572, -0.19831228168574028, 0.13894573572439178], [0.4905599140921571, -0.01961777545987433, 0.43383646713704993, 0.7276655877062017, -0.7359958998687868, 0.3777327802396756, -0.16243175036981083, -0.10512969789286422]], "dtype": "float64"}, {"__nd__": [[0.47343096391331535, -0.12833483557768782, -0.32499335464027845, 0.6201272277427786], [0.5146735044368127, 0.19957046886397287, 1.4029066223502946, -0.49617820223647346], [-0.8144155848832798, -0.10980960098321496, -0.5690909072892713, 0.18303381064827626], [-0.29552774116373726, -0.299964643326548, 0.003365692077281758, -0.4204495431696546], [-0.958643356797496, -0.36338153212411833, 0.35003303766427757, 0.11702325289344911], [0.4647157954012529, 0.8390044547128541, 0.43760390427182616, -0.029398770779724057], [-0.3747692735843013, 0.23067897633469758, 0.8007213764349116, 0.7094361236956321], [0.4664314618451409, -0.4003100734588467, 0.29052101537469116, 0.35313271489807496]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [0], "dtype": "int64"}, {"__nd__": [[3.311511316345457, -3.4974094502407023, -3.085370939894265, 0.5697443110661343, -0.22643173062797492]], "dtype": "float64"}, {"__nd__": [[-1.3574474881236043, 0.4273612754461392, 1.069739876317004, -1.9913350188365042]], "dtype": "float64"}]}}, {"a": [{"__nd__": [2, 7, 1, 8, 2, 8], "dtype": "int64"}, 4, {"__nd__": [[1.223223107041293, -1.2633093172353846, -0.6939934116076121, 0.13807909570420004, 0.12484216392454832, -0.6069092765812635, 0.4997208257260341, -0.2927541489449597], [1.1051142347629395, -0.4851699780914362, 0.19844459511866253, -2.0906994486125945, 0.2031612052744749, -0.662191797819848, 0.7080321206282764, -0.17094871473373702], [1.226693549998587, -0.529468538289035, -1.4316061840807277, 0.2394268642869366, 0.5795039549335359, -0.7422054675024669, -1.2922992645986542, 0.7551930882205851], [1.050581245866634, 0.7304855056610187, 0.09088828726391032, 0.0012510552822948323, 1.62978654447218, 0.8015752609485829, -0.9025446372968787, -1.0707812369506258], [0.6260390559739241, 0.7324926339335227, -0.6077111829802501, -0.7525943498335237, 0.9662163974674023, -0.31027585418496095, 0.8371483380315172, -1.097266419876259], [-0.9802282874682008, -1.7000600308796243, -0.02699499480402971, -0.3967001864849173, -1.669042426724841, 0.1587799886973623, -0.5231709873577696, -0.2176149197081831], [1.397121848646859, 0.9693170517899208, 0.4947219527496488, -0.3268887450232853, -1.1829593412708606, -0.6518041514510096, -0.3717904195397754, 1.2549564494255732], [-0.920362623187538, -0.541000063342038, -0.008453300836832583, 0.3904193981964983, 1.0798059128397337, 1.3245890570392842, -0.1478435097945767, 1.921690385981471], [0.6709038062725345, -0.7827560615183228, -0.9739353207194392, -0.4172091409061384, -0.25160077017980537, 0.7862079124482242, -0.8154555337854502, -0.5698144659771811], [-0.8488834315940028, 0.2645783332290238, -1.4512047054396702, 0.5469889839077605, -0.791116044309921, -0.21879202556888716, 0.4285134221720177, 0.29454676487041226]], "dtype": "float64"}, {"__nd__": [[0.8318308909672225, -0.2729193524398181, -0.6247684336345856, -0.3931551931591638, 0.24685949272038513, -0.32104021840965363, 0.306141855347864, -0.06614169754302199], [0.22128955221179944, -0.458780242209376, 0.3103189808601195, 0.3451249888710281, -0.279858041862174, 0.3939878733339524, 0.7872281013813364, -0.0012407898145711926], [-0.28116627299835045, -0.1796488103262729, -0.08693034948660844, 0.3016449935614595, 0.5470417633402896, -0.1251521878826014, -0.06804840970188224, -0.41578736768909824], [-0.15700015466593575, 0.1406820987669948, -0.601585299361276, 0.20590837201827092, 0.2676527643098857, 0.3609986794205881, -0.17133320982090755, -0.4698602430259154], [-0.0838462134945009, -0.3103666430778768, -0.5280415967914985, -0.22815029036123802, -0.41744900488882725, 0.5763059348965153, 0.22340670149636663, 0.26314309642894934], [-0.33274358245612395, 0.4105548579050118, -0.1127076733773549, 0.2546336342089514, -0.30572762963476346, 0.18664944706300882, -0.5021643709044357, -0.05229219945335322], [-0.07309958677258085, -0.40974631478342066, 0.0076208330175199, -0.17079927274492016, -0.11888595888566295, 0.2204137700081212, 0.3874747200102507, -0.0598240519037572], [0.38409979502347974, 0.2996410774346772, 0.2228646436800443, 0.47526904816985205, -0.450648461540134, 0.39832038711431444, 0.11300500737822182, -0.11689778091803138], [-0.05001272291992578, 0.3050553779241917, 0.360126810923132, 0.21552404544820766, -0.34610120355062474, -0.43648866633346706, 0.014947562163109641, 0.19229938762207544], [-0.2042560639883836, 0.26639689482226064, 0.18542542127600728, 0.1479007356766515, -0.22465159540740723, 0.37590574079036815, 0.29396764022570393, 0.4099185672593744], [0.11794911531297464, -0.37370088886441916, 0.11867153467557637, 0.2642257080931444, -0.03679839363831587, -0.3371873572605793, 0.17184763697932307, 0.19682243934872912], [-0.3703943438482284, -0.21299486499983392, -0.42940423962004143, 0.10089752021329843, 0.05626753166292692, 0.522337674723012, -0.2716873968335479, 0.39981012299842317]], "dtype": "float64"}, {"__nd__": [[-0.10663291279348656, 0.13656870652230566, -0.6139804700288377, 0.013728171044499384, -0.2628144729787186, -0.054336171564347745, 0.0926343443237984, 0.24293685129960665], [0.41663087782434016, -0.00950750209216181, -0.44380808909312963, 0.5126070833544196, 0.04222657026253786, -0.29994035000466807, 0.6854481961095483, 0.14633234707597745], [0.3634430000549554, -0.8416789437442157, -0.3077327960385383, 1.0072877415218637, 0.25321401958464823, 0.24378665144479353, -0.3501913362928433, -0.06588868166829558], [0.18676813418516797, -0.035276159930645874, -0.054678256495455796, -0.8738682002300363, -0.3968522863752751, -0.2248587392873889, 0.2806077653107387, 0.25657211396899277], [-0.008444436089873112, -0.36342842697345035, -0.36389203981547746, 0.11956563113525076, -0.6782906581288282, -0.1149779590433791, -0.16161959440341392, -0.13300553212928348], [0.8192988678520151, -0.07921165812485569, 0.421539509177023, 0.7866659948768504, 0.24849193898722424, -0.0069770437556060366, -0.6324381588345577, 0.7987260690953062], [-0.5611749842255574, -1.0263639246760647, 0.520272438498221, -0.7410419036584663, -0.6092344624539232, -0.3696025752143732, -0.6069889354326177, -1.1029441272849012], [0.07610551148850277, 0.22030087082101077, 0.35509929724988315, -0.03777141602802255, -0.2230559343268922, -0.28904276451431254, 0.580676042972531, 0.39348338725764465]], "dtype": "float64"}, {"__nd__": [[-0.2449163101612532, 0.1268629590207065, -0.6149072382916789, 0.2218881883909986, -0.5629409587217085, -0.10151857696565328, -0.14585115353502243, -1.364747382058515], [0.3032591519117302, -0.04577649794239447, -0.42164966956961314, -0.3905606748828478, -0.08541939537000896, -0.11725451861486506, 0.18736461354530265, 0.0376281438568516], [0.26545022765506265, 0.08509109698150095, -0.7283015010015813, -0.731696441167712, 0.19793376282176428, 0.16106034061422497, -0.035879248236914595, -0.6818029702239612], [-0.3826240641386449, -0.4340014957194991, 0.2980984657255434, -0.4314880726993264, -0.7032503214032245, -0.4254264649296413, 0.12228976430347896, 0.7530141783041456], [-0.2746814328648868, 0.5263136768400324, -0.24974846118277216, -0.24110162073190738, -0.44841265523024265, 0.3655173736994555, -0.26493380894157337, 0.44464361231642574], [-0.22295446808531347, 0.8407978747612914, 0.08893576936043582, 0.00878303074601603, -0.6059943475222864, 0.623827516635621, 0.1360362760599236, 0.02650094911549566], [0.0006051216805719535, -0.25399352992065877, -0.2647605489829759, 0.16401352463626243, -0.4022951424567219, -0.07537160281960921, 1.1158179648327087, -0.06613530865042797], [-0.14268920386442296, 0.44287270918185145, -0.4854548582557738, 0.6910835390129993, 0.0020878263168745896, -0.5778654880374239, -0.07147973114064851, -0.3440330515825919]], "dtype": "float64"}, {"__nd__": [[0.3421898016332706, -0.22970112265374182, 0.35678731913265643, -0.9530151897669225, -0.2173669474454954, 0.2806137230790505, 0.24944066790750533, 0.02986851198636558], [0.3708508134305262, 0.3849578865089706, -0.0016288355440345638, 0.10411221333290799, 0.44477107668580357, 0.08202672064913041, -1.2695421944421548, -0.7818097291483772], [-0.32522502297024186, 0.3377505878035045, 0.03104431738782876, 0.3197025503448071, 0.49630985214109496, 0.2677608475554427, -0.4698761197826695, 0.41317869437872823], [-0.6732321757550682, -0.6423658344811222, -0.22730875948946472, -1.1367071453080986, 0.8145205358202989, 1.1557103497188594, -0.10284097691604946, -0.21045936427178852], [-1.0601563853400497, 0.23598576876127744, -0.37604449517686545, -0.6298043336504177, -0.721528556310163, -0.09562320305508437, -0.14558166727913832, -0.10389198799723107], [-0.3456955460547634, -0.8986533279936433, 0.6692319854424603, 0.36068304117491035, -0.3357832873652657, -0.7156721663740326, -0.5458195915699945, 0.29826376292110107], [0.20716898664143427, 0.09603430472392542, 0.548748812559161, -0.3181107063883016, 0.6669609317950269, 0.04050711948155345, -0.41574649330935676, 0.21201393013143296], [0.11745907299149524, -0.38872703916318413, 0.8491318928776093, -0.8559983837612305, 0.08784617233497498, 0.1633280623611518, -0.3927028928049785, -0.011800926013070005]], "dtype": "float64"}, {"__nd__": [[0.6141826666927683, -0.5913899036590281, -0.5736504466072878, -0.20310102360985638, 0.21840886917678315, 0.523100797130627, 0.25631388414690326, -0.5891749594180371], [0.5213310772948866, 0.32258271049233045, 0.4787466420372575, -0.8565380436880943, -0.8783318251049967, -0.37695597389490554, -0.2691282927338534, -0.053720843138414444], [-0.5941583779426318, -0.5830952999340322, 0.1589133209725398, -1.1093764266118342, -0.6430723281845425, -0.044226754127889824, 0.3153817576100043, 0.08003935139064401], [0.7357119645955555, 0.09421294601746556, 0.2948185037607159, 0.3471705711438485, -0.036427556868377516, -0.7125324193151477, -0.2006080795303006, -0.16446861572188604], [-0.4957882275870751, 0.2662777336922189, -0.33029105454516394, -0.8083843146342181, -0.07621613231280118, -0.5723889196590342, 0.18987924399101044, -1.3735982551043933], [-0.9185109266152247, -0.07075122785455833, 0.049093935427332994, -0.03487679599760331, 0.6777598961159341, -0.19795873063726446, -0.32969445863779945, 0.33737813450562093], [-0.23544555184417582, 0.12190870592699331, 0.5533472146696523, -0.15241056479938955, -0.12513355499594164, -0.37212071031317145, 0.12535072047596155, 0.24077931726858018], [0.4482182045293241, 0.26740749084459947, 0.9293540516859177, 0.01940595109299734, 0.009617370059107868, -0.19361690180817295, 0.5809166998590114, -0.008189763778484401]], "dtype": "float64"}, {"__nd__": [[-0.1892968712092941, -0.6899196486757485, -0.10866781369883313, 0.3834592142961821, -1.3199634451618532, -0.5907693520233548, -0.49357142812896854, -0.4442638694254632, -0.01611879667237388, -0.22638688655071731, 0.029246851406253667, 0.10669693569323056, 0.6331843684360959, -0.15764005325346586, 0.13845878943279644, -0.1425849223727927], [0.0493003013321618, -0.19902428278111153, -0.5968878342546691, 0.16730138020043178, 0.5080968083655957, -0.4900729644973592, -0.04179489789518244, -0.04239226491339201, -1.0733246958221745, 0.9306430231532041, 0.6952998385540218, 0.5558705717172039, 0.465456630805119, -0.44742518429845407, -1.2332986441242522, -0.056687309717770425], [0.014955297374999003, -0.15978313264850594, -0.24358773917259807, 0.46025957395739137, 0.3572943926303814, -0.03700759387166182, -0.2753446764045642, 1.0956357991337622, 0.38155740483185857, 0.024201163863321266, -0.6702904631116253, -0.7910104566784023, -1.3926212581643735, -0.1525910847090984, 0.11619221271064331, -0.11503975511816769], [-0.028698727990654507, 0.09251537310173574, 0.38552424122739026, -0.9035112056454142, -0.6426689651928046, 0.41156048813212687, 0.1112172728002073, 0.2212083677971061, -0.5219100908122954, 0.7194896069512465, 0.03402348614188185, -0.13907284651064805, -0.4600356598730756, 0.07186365286253925, 0.060985307168342856, -0.07789788203384909], [0.7818572910495137, 0.5703370067021516, -0.02141110879302651, -0.3018900711654761, -0.23215111025761698, 0.15579496172521126, -1.3877842736015942, 0.1415635066248716, -0.11888464958225262, 1.045851707511174, 0.21138428116709485, -0.08850903764267667, 0.021431200874198312, -0.31466351893690575, -0.3461149770146244, -0.03249358682473768], [-0.9925941288823237, 0.003090284532473291, -0.8303835547783571, -0.35415806103058417, -0.36045141680392084, 0.3470537780522511, 0.7771559943668493, 0.708582931071236, -0.2364906961146745, -0.11066549505666434, 0.022073428911923793, 0.08153224268017951, 0.5605923046000638, 0.34034587764017016, 0.0024121698314249263, 0.9611959240815139], [0.45396168879860915, 0.9204487192565783, -0.01915127672693925, 0.29620001036335875, 0.17677286644437984, 0.10076369996383946, -0.5052043035772686, 0.13962483609908324, -0.7089509269435248, 0.20628012403096288, 0.870900011467867, 0.5812870998715485, 0.16550108109967843, -0.1266110066397391, 0.396890825034911, -0.4996435973774274], [-0.5823657590601917, -0.6301419755466698, 0.20969763850804973, -0.6390023548235406, 0.3081844617656148, 0.5193012698678598, 0.35397962003388556, -0.038131742159838285, 0.22339693927283039, -0.5815415249159547, 0.11238488269263325, 0.8442695420610916, 0.7369085855000528, -0.7731364068437453, -0.024547551015232087, -0.15632968735478356]], "dtype": "float64"}, {"__nd__": [[0.4623027436881828, 0.14454106463443261, -0.03995170803327595, -0.3088087337214244, 0.08434671051495854, -0.06359159537984004, 0.03758679868475717, -0.06774025791271315], [-0.6043219121387551, 0.2097993334068275, 0.23014159889116745, 0.11472097540027408, 0.13462810822101648, 0.6104085660057352, 0.5453010595484639, 0.8280698096890848], [0.35186696901995335, 0.427157139534761, 0.034878490443519254, -0.5076687643013069, -0.5717629768464241, 0.2651030789241267, 0.07104429152641337, 0.2781905807820839], [-1.1370354857796114, -0.10617194174347704, 0.052495919401136876, -0.1918916682572146, -0.14241296034751252, -0.9050141034025082, -0.0726057344281109, 0.6053573296927739], [-0.5878505549779406, 0.2485601614331663, -0.38257407310303565, 0.06425459272358584, 0.683186916137284, 0.0036157487051796675, 0.16147716623437974, 0.29773085197797966], [-0.03623004274386506, 0.6031959754999934, -1.0125526668406581, -0.17839845809799804, -0.9660830787899499, -0.7738711134737137, -0.025034201501912424, -0.619573025438902], [-0.22145665571822937, 0.6525090709830293, 0.2514351331179058, 1.8461050414284514, -0.06784971457398746, 0.01814498982295028, -1.2425730275552134, -0.0259964346512166], [-0.2431176656996259, 0.22228094448502148, 0.004132955932683526, -0.3358958978439072, -0.9879540934527125, 0.36472634181292524, -0.3673102910818082, -0.8781580732348058], [0.17400206083732073, -1.0193275093602854, -0.33368314321708875, -1.0599245749327206, -0.17695139481470296, -0.877080540563115, 1.417243940103671, 0.5244361102870096], [0.19748014906166023, 0.13509366097050599, 0.28883397085102464, 0.3851577295706186, 0.09791133471248342, -0.5860236194887102, 0.04232457781445785, -0.8231026852562062], [-0.22156587646009007, 0.1619020934950065, -0.5857632851537476, 0.1798588581320298, -0.2820177969173487, 1.1528944087293835, -0.22937639770564017, 0.2097303514021987], [0.27143204889867223, 0.032486671434933743, 0.12427982120742247, -0.16846762557451936, 0.0340142515658859, 0.05670474078196105, 0.3841392154032475, 0.6093312012955426], [-0.02989118572019525, 0.6885888894440008, -0.01733459182424357, -0.4858110226969077, -0.22800838854379232, -0.3400617081807545, -0.11833636309740259, 0.028719170013252273], [0.1076767312791241, 0.5858013925666019, 0.3718900773106313, -0.21411922492805888, 0.27577673688182325, -0.034131689377419734, 0.3876305659537572, 0.6214063309379954], [-0.055013861540044194, 0.3859975337565835, 0.6908241085669011, 0.18941896901979538, -0.5431222833688746, -0.05194662673941137, -0.4439706374420303, 0.5016414009831494], [0.10442532624445466, 0.16352943657419808, 0.8907830470496694, 0.3774659121873807, 0.07304828823168838, -0.900640297325134, -0.25951198989003355, -0.3446797313671789]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [6, 6, 6, 9], "dtype": "int64"}, {"__nd__": [[-9.150976358505657, -20.254992145210828, 1.2626754671798182, -3.9686537987222383, -10.725454505199176, 3.476661904849852, 10.219285951569372, -9.313640808642958, -2.9706994100756052, 8.561315391804312], [-7.894385865507017, -9.418550391226692, -0.8657994708211594, -9.476243218062804, -9.22665020818324, 1.5716113641637879, 13.508718318939723, 0.06526722310425015, -5.220472346489316, 7.102876146568013], [-11.869302411106245, -17.370767510655057, -1.900710838069754, -4.410544659760884, -9.974506286047923, -0.49655547330369565, 9.242573076764327, 2.1297563742398875, -4.907844386281873, 7.726255391795341], [-7.953680080993496, -7.763717270750598, -0.5374790592030858, -9.959966628813888, -6.891749025250256, -0.023771741627840015, 5.482725573396394, 6.938101400010208, -7.772310335206607, 7.572099105796358]], "dtype": "float64"}, {"__nd__": [[-1.3233908932635203, 0.2862114911298478, 0.1500082111100572, 2.446990179920647, -0.717270700029548, -1.0657776351715766, -1.9087924480301401, -1.3858519170510795], [-1.2098205509864144, 2.21765318247689, -0.2976806379577902, 0.9513905495592183, -1.6319513273122905, 0.1304336415557163, 0.5917751615329473, 0.9627522998129461], [0.3643092976080226, 0.2313465439903017, -0.9828549608567524, 0.6445293951257134, 0.7552180767556096, 0.8472419272321388, -0.017337785454462074, -2.100472155395849], [-0.7703825308459676, 0.8154409568306379, 1.9031894045365183, 0.7346315944433246, -0.7052922052047282, 1.255948661817739, -1.0395187483394326, 0.650056970057145], [-1.2125711290339198, 0.6800759412786306, 0.8153640335325703, 2.498506246247616, -0.5074249915893264, -0.8901206417437297, -1.583333408651883, -0.4587038142240674], [-0.3977832519070088, 0.6218321503025381, 1.568513760108865, 0.5825253722641721, -0.07018281477666694, 0.7654486341824482, -1.2457086952335759, 0.22208182279677724], [0.45244486510638293, -0.12148685455708014, -1.8618783159386767, 1.0610194524832142, 0.4976548522287145, -1.3441560815383045, 0.05113892357201263, -3.5056570279516497], [0.4249743721269386, -0.3386282471312826, -2.2072419926342506, 0.44623544828608724, -0.08080581269825955, -1.6705502648455526, -0.001427046669567427, -3.8681950152081215], [0.7819872898506955, -0.754579467215574, -2.344478257196146, 0.5244986859605821, 0.8720488299491086, -2.1483513866712856, -0.24656381293760246, -3.6402407336922047]], "dtype": "float64"}]}}, {"a": [{"__nd__": [5], "dtype": "int64"}, 8, {"__nd__": [[-0.18891308948146338, -1.2418675452121695, 0.6231746537438295, 1.2492834497798793], [-0.35829185915167566, -1.46510614330511, 0.10252865048972155, -0.2410863930274411], [0.9897311588678164, -0.06414718306064189, -0.7486068103433263, -1.3904842097448644], [0.00418785330543149, -2.341910984263244, -1.1102516594402094, 0.7103474889491006], [-0.16345555425083433, -0.05298767650404247, -0.2461647315508851, -1.3935243195851164], [-0.3969093199015442, 0.4505880828440237, 0.5090497571521214, -1.0477505563543439]], "dtype": "float64"}, {"__nd__": [[0.020600839905206978, 0.27385631959457374, 0.07293833983500951, 0.45160462599265366], [0.001140335081828753, -0.492889969343203, -0.005774114983760804, 0.045095711217567065], [0.1988100688504856, 0.14885003407085232, 0.45303463060232135, 0.17949371875468348], [0.05769680606143898, -0.15668499140801134, -0.07089946249296207, 0.3902337366100983], [0.5691215868917173, 0.10625657224405326, 0.060132640861944836, -0.058229038844894006], [0.07312407519976223, 0.01846193916360637, -0.1462931025976042, 0.2709348692074895], [-0.3265474023847204, 0.045698976662612065, 0.05068362046843123, 0.19780923227176744], [-0.04587201282570395, -0.2836219459160483, -0.028485036075470562, -0.30660039799917627], [-0.10588534788445933, -0.22816326152023397, 0.7243075108169408, -0.6485471801289592]], "dtype": "float64"}, {"__nd__": [[-1.1992645554078847, 1.2234106120306143, -0.20800891432642177, -1.3707827980097076], [0.6209271756305005, 2.09519420926836, 0.032124526717779676, 0.8696923419259699], [-1.3355820487857468, -0.5807186450435844, 0.2017858306843342, 0.1798940267670293], [-0.6717209522905624, -0.5813283679593971, -0.616801699717753, -0.7839785684683802]], "dtype": "float64"}, {"__nd__": [[-0.5484614892467495, -0.5689804769994137, 0.446039467060252, -1.4159428434426353], [0.49015129369678156, 1.739110753866204, 0.15974962922807984, -0.4689913869709015], [-0.9315867724971494, -0.5179258400287515, -1.2062557584064257, 0.246588549739183], [-0.10384680510048024, 0.7548412464105937, 0.26975369811948713, 0.8013891346642098]], "dtype": "float64"}, {"__nd__": [[1.511719804999348, 0.9802355575016852, 2.7696456760886554, 1.1745568451173047], [-0.41266049615348, 0.6602440859430362, 0.81698332468367, 0.17481911419674082], [0.808191045922329, 0.6487068393691633, 1.4043909352288213, -0.03805595204103769], [-0.8511533588796988, -0.3525236759403814, -1.846166872233831, -2.586589586222547]], "dtype": "float64"}, {"__nd__": [[-0.988546397662261, -0.4793256572467963, -0.6566230985127698, 1.2159136706171545], [-1.0604566529599728, -1.188367818595105, 2.0067546554657345, -1.5081034087597416], [0.7925783415034172, -0.08100622348978213, -2.5263479687581514, 1.0106369746643176], [-0.3006312992435416, 0.40872962489624565, 0.2559392466566333, 0.3715325239735535]], "dtype": "float64"}, {"__nd__": [[0.937714054015126, 2.212381107137478, 0.30695539489357443, -0.1508513191886279], [0.9432128551857792, 0.5070323292661385, -2.4901311247357087, 0.4875846591430144], [-0.35604800608352266, -0.20343689858683325, -1.4140553807006226, 1.9355500044737188], [0.5295946008822103, -0.42058772803078703, -0.3923244037349633, -0.1590702369155902]], "dtype": "float64"}, {"__nd__": [[-0.08067242128696514, 0.8139740526615704, 0.41458773855855496, -0.4845865204732599], [-0.21717733884443857, 0.6328940594669118, -0.9514304055494563, -0.38622396135031656], [0.9849122263597995, 0.029221597772020082, 1.470081093005204, -1.1047865214475814], [-0.4134217820446244, -0.7660425749915964, -0.8202817899717333, 0.8061730116541087]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [2, 2, 2, 2, 2, 2, 2, 2], "dtype": "int64"}, {"__nd__": [[-2.2225899375370135, -0.9533353664519062, 2.027945993884179, -3.236183560402898, 1.441162260492828, 1.6424380781787598], [-10.649114888743018, -2.882397860262431, 15.854956541151157, -20.89151464132653, 9.608721666327597, 11.669575284542757], [-12.059653674357028, -4.185110811326474, 14.917019152696202, -19.3854536184052, 9.20135134125512, 10.365837885372478], [-11.661740979547773, -4.030083520910807, 17.07259519319745, -25.041996740573623, 10.013224930958472, 13.144095054385414], [-15.614761208246852, -5.747012044250846, 19.170646169813526, -25.85272986672672, 11.711734879482172, 13.530237290627273], [-12.651396207357859, -4.7129741475094855, 17.51125928204928, -26.067263989071023, 10.322229737050392, 13.45416812523445], [-10.72939416527808, -3.9892569963835185, 14.909736913886427, -22.538338443999027, 8.832705285748158, 11.646796521748783], [-12.638882235928497, -4.108609826959944, 18.666828190029694, -27.06725090651898, 11.097159815983101, 14.439265252273614]], "dtype": "float64"}, {"__nd__": [[0.08121350775935776, 0.7225790181707655, -0.9149577718305683, -0.14116015637001564], [0.02599825253817634, -2.157377194004249, 0.9000341268463199, -2.4059731968783926], [-0.20924325332181726, -1.2899702177345604, 0.573534025339243, -2.765990689906713], [0.1845989421585402, -1.3106045116223324, 1.1506292497531416, -2.383200467605209], [-0.04251305500302819, -1.5506938389767893, 1.1417169823822784, -3.557748233825964], [0.3446106978423857, -1.0657858074932314, 1.2442527797371727, -2.601383025315119], [0.40125833488298496, -0.9482298964053902, 0.8130044160567573, -2.0582748793273216], [0.21203541828540806, -2.0203999724391415, 0.8450191066623113, -2.724996993586268]], "dtype": "float64"}]}}, {"a": [{"__nd__": [9, 9, 9], "dtype": "int64"}, 3, {"__nd__": [[1.1120715404400714, 0.15595918062108516, -0.05185655965598293, 1.8509549913651506, 1.6825100778898776, -1.9854835987807657], [-0.0036879734454724144, -1.9463562814166917, -1.464695116642166, -0.5834817566526397, -1.0735090794164543, 0.6736616073927093], [-0.34390212045774143, -0.15754597145090576, -0.007964920723567004, 1.8153319117608178, 0.3829854603185151, 0.9934776673938464], [0.6037357036206406, 1.2935525401885035, 0.12952021536367253, 0.020334628108299897, 1.3615229582992723, 0.24919365145578698], [-1.5153331399992136, -1.1802252839292504, -0.3561599985278383, 0.5753349816983261, -0.6934201079464714, 1.5747110615318618], [0.498045056497055, -0.1829458967991039, -1.140303188972525, 0.7791982068318923, 0.6921030916004454, -1.0838609066366927], [1.3912487119419543, 1.2913785700569964, 1.5427930056496155, 0.03739786889364022, 0.9968691393175174, -1.9821358160239835], [0.47722906826021094, -0.3579479805098296, -0.0430921239667274, 0.5005594088909417, -0.6542813403601672, -0.14153169376900354], [-0.947636236843821, -0.7524847852175843, -1.086373176603313, 0.3781025108485659, 1.2850745441118652, -0.9349140322852776], [-1.5641453540255577, -0.19115717824936715, 0.41853776155736033, 0.27368103161569957, -1.1483001696816955, 0.014793455630331164], [-0.8309820472703866, -0.40998815644802983, -0.12410873867214033, -0.42377558703791185, 0.4590363403587066, -0.5017642683416035], [1.5069924427422559, 0.119850321176369, -1.119201555040754, 0.9781337885199607, 0.8552753470101706, 0.2674668685847507]], "dtype": "float64"}, {"__nd__": [[0.0006957505913995661, 0.1024669852394678, 0.25624578324886577, -0.036070912077645456, 0.24773954009733826, -0.1680456159388887], [0.14993870713024693, -0.2945542421413829, 0.10504995148335368, 0.1720228831686755, -0.15841160478061364, 0.11006385066628235], [-0.5507787947613804, -0.027236870793129874, -0.07558682754959613, 0.18783371460046852, -0.009650935803397506, 0.3047198424237299], [0.5675604107348219, 0.03296556563705853, 0.26427130533080634, 0.5006521873508879, 0.2876321019397785, 0.3304242488634661], [-0.05432730640131237, 0.12467454389334828, -0.28684350173643935, 0.26886584555084425, 0.1394328737575813, 0.14087495907689618], [-0.5342229655475493, 0.31895913993899944, 0.46895041628343137, 0.4311816834646225, 0.15627956050590777, 0.4154167206146471], [0.03539872456046907, -0.16656204238463368, 0.010871934365050634, -0.18611745764244783, 0.541942466281344, -0.6985729624631368], [0.0964690374352983, -0.22748862145162446, -0.13944776161931188, 0.23163058533731917, 0.0003218619883980578, 0.34194065144659896]], "dtype": "float64"}, {"__nd__": [[-0.037754518100290416, -0.03133947971425975, 0.07627485755368535, 0.6355321310949346, -0.3095829284802231, -0.1305774708323373], [-0.2497646600495462, -0.20624841019208598, -0.5849109199726069, -0.13551402043529986, 0.22801085761803097, 0.7014425262110187], [0.23956674453893367, -0.27423621320872027, 0.7150750634223181, -0.8314874412531592, 0.2469497269267064, -0.5824334082642459], [-0.05682022101207042, -0.647752984329865, -0.7369307486854907, -0.2630333051443324, 0.8715089308420509, 0.17893176992730356], [0.4441694478989995, -0.15592287619465964, 0.0175851151507529, -0.3656946597896759, -0.2772265567810744, -0.1510815511757861], [0.47093175976949225, 0.16774353190482236, -0.34916224399484014, -0.07447329635310147, -0.4135682819190498, 0.2704054425597423]], "dtype": "float64"}, {"__nd__": [[0.1390721404817861, 0.13420590091823215, 0.3563954702230398, 0.28801785631991045, 0.7259040804312475, -0.34523696328244524], [0.3782581009618237, -0.06473848549557319, 0.4725211711589961, -0.22194821561142425, -0.7785319547704098, -0.18174016777156107], [-0.28235177961134145, -0.5024218143662059, -0.15883073799607716, 0.5829117908489615, -0.13846291301770094, 0.3551555211703193], [0.691323878774109, 0.5242963338636745, 0.2526895204136564, 0.4735262049513519, -0.18162208573975008, -0.6057053372997057], [-0.7078436684464833, 0.4768760718279433, 0.6851037650243657, -0.12046199216397814, 0.036860448503454386, -1.0977060895604656], [0.3746899906064561, -0.029609831916646095, -0.8799047091795104, 0.2627193866407509, 0.3142288503885775, 0.42931261682579797]], "dtype": "float64"}, {"__nd__": [[-0.030704698932858727, 0.1476404813693394, -0.39919984173572143, 0.7073502085095962, -0.13699578889518044, 0.6402648748932632], [0.29035958594390227, -0.08027587579712737, 0.6690833875148932, -0.28542551949200523, 0.21530200707946065, 0.10590496548609185], [-0.025367361470468176, 0.13049105542099804, 0.3836471323848209, -0.1433914340170463, -0.2013379233762129, -0.05680721416283205], [-0.2293123475755, -0.44206615056141035, 0.16766147743007945, -0.8110201029024156, -0.018783153444154863, -0.21089835907023838], [0.08933326563345262, 0.23461316235289953, 0.12988082277729637, 0.0015508712651628783, 0.6468071738527659, -0.24820015703361803], [-0.5756167086405267, -0.11050786832829014, 0.3450766584901954, 0.882483172429895, 1.023626966867897, 0.19455528317937137]], "dtype": "float64"}, {"__nd__": [[0.5814083927413545, 0.808943628021267, -0.20824385836944864, 0.7513551251989184, 0.24467474781533935, -0.027369568770060766], [0.29817441274062884, 0.3501269711624436, -0.9423055697589697, 0.06832904609700222, 0.13420299930296767, -0.3128013472027277], [0.11233055863301196, -0.16772227521516242, -0.36824558884248826, -0.03870815809382577, -0.9472926696543047, 0.2218489575439028], [0.2754256226860928, 0.16699118151822756, 0.2824262406948276, -0.19736504888115483, -0.2631536957617605, -0.8436784050797435], [-0.833312768681772, -0.35737223134537, 0.45123688309131427, -0.006597673096074039, 0.6548624890702476, 0.37846948912534656], [-0.4895868789170994, -0.2834814722292045, -0.5266256452500869, -0.7028699178798686, 0.12377534026184796, 0.32754503171655935]], "dtype": "float64"}, {"__nd__": [[-0.4358445747617992, 0.5174576424836635, 0.43286879651744314, -0.2702460131004824, 0.17416188765185114, -0.4603063738063852], [0.018018898591627912, 0.26562806362476643, -0.8907565970172492, 0.7253294013221252, -0.8151054030252086, 0.05710552578007607], [-0.2220633508041181, 0.44440688085131774, -0.41003910637803936, -0.2199197011536719, 1.5418646069929962, 0.8522479794813986], [0.4469234236777058, -0.32166702201980213, 0.1943658425189346, -0.7847073506833923, -0.36405901339376623, 0.23308142544811475], [0.4078284591370574, -0.1480142890594782, 0.3791363480146786, 0.430128603329031, -0.46235357211624784, 0.13978555742346085], [-0.006153519831328798, 0.7196073638738508, -0.5928772944958076, 0.2375748425683488, -0.06176800148088994, 1.014416063984569]], "dtype": "float64"}, {"__nd__": [[-1.110774116180297, 0.8512641242075284, -0.07982493218349489, 0.27687840725985186, -0.7229135793375722, -0.6331726051939914], [-0.11468820218987484, 0.046124090866312285, -1.197175340048485, -0.3188738438918914, -0.22061847295451079, -0.07611464717180805], [0.33629093002171523, 0.155862672611811, -0.12718416656226952, -0.6038174665740321, -0.7958751479313324, 0.04916957817066636], [0.7229309569589427, 0.006933170294010808, 0.4587581226629811, -0.5577341584966627, 0.45923668009798446, -0.09739111438364492], [0.058151460352102065, 0.33333507942129925, 0.3827037564046989, 0.13746315285474334, 0.19935825136395952, 0.1702950395096341], [-0.3423081904203673, -0.1432058803420018, 0.013755723995551177, 0.8147118177913422, -0.5764029326464497, 0.056233695962776456]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [4, 4, 4], "dtype": "int64"}, {"__nd__": [[-5.8583470524956, 1.4670379606917325, 7.637106515824337, -6.2344460257062195, 11.28010898381606, -4.040021451700103, -9.163159863665015, 2.401441022519241, -2.7520406089371483, 10.276881216444098, -1.4013813444631034, -4.9671195850260474], [-7.080151134480355, 4.145955193946551, 11.780935390258879, -7.570734523485353, 16.47828589241741, -4.521291376179894, -14.379678287501456, 2.958312223171839, -2.079100741082197, 11.474873639087392, -1.7304916241870127, -4.0180740401231025], [-8.113584522172951, 3.021462526395467, 11.514756892984867, -8.417658393744002, 17.52172847306204, -5.641910131293389, -14.40642561906172, 2.2857236678584845, -1.2532120363912131, 13.791748957999385, -0.44479781549688235, -7.048087858316789]], "dtype": "float64"}, {"__nd__": [[0.30279323707672334, -0.8434483968901443, -1.1283793121284995, 0.14345814348330296, -1.283805356986519, 1.5743683489611358], [0.7516192505951257, -0.8145681975126844, -1.709158574787378, 0.40695548946426136, -0.810815882827805, 1.9804877993451036], [0.7848538043832917, -0.7616922514260934, -1.8692543260656107, 0.08121715397647059, -1.4387948804098907, 2.0203610044934197], [1.2051010971077676, 0.3074568489896656, -2.547745404507382, 0.9869999073970983, 0.6061773523998919, 1.1146779955091621], [1.182551982516051, 0.30836597615518163, -2.631831480587917, 0.3245763713263299, 0.13673008584599528, 1.3386754688222644]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(generate, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")